In [1]:
import pandas as pd
import sqlite3

In [3]:
column_names = [
    'status_konta', 'czas_trwania_msc', 'historia_kredytowa', 'cel_kredytu',
    'kwota_kredytu', 'stan_oszczednosci', 'zatrudnienie_od',
    'raty_proc_dochodu', 'stan_cywilny_plec', 'inni_dluznicy',
    'lata_w_miejscu_zamieszkania', 'majatek', 'wiek',
    'inne_raty', 'mieszkanie', 'liczba_kredytow_w_banku',
    'zawod', 'liczba_osob_na_utrzymaniu', 'telefon', 'pracownik_zagraniczny',
    'target'
]

df = pd.read_csv('../data/raw/german.data', sep=' ', header=None, names=column_names)
df['target'] = df['target'].map({1:0, 2:1})

conn=sqlite3.connect('../data/credit_risk.db')

df.to_sql('klienci', conn, if_exists='replace', index=False)

print("Dane zapisane do bazy")

Dane zapisane do bazy


In [4]:
query1 = """
SELECT COUNT(*) as liczba_klientów,
AVG(wiek) as sredni_wiek,
AVG(kwota_kredytu) as srednia_kwota
FROM klienci
"""
result1 = pd.read_sql_query(query1, conn)
result1

,liczba_klientów,sredni_wiek,srednia_kwota
0,1000,35.546,3271.258


In [5]:
query2 = """
SELECT target, COUNT(*) as liczba
FROM klienci
GROUP BY target
"""

result2 = pd.read_sql_query(query2, conn)
result2

,target,liczba
0,0,700
1,1,300


In [13]:
query3 =""" 
SELECT target,
AVG(wiek) as sredni_wiek,
AVG(kwota_kredytu) as srednia_kwota
FROM klienci
GROUP BY target
"""

result3 = pd.read_sql_query(query3, conn)
result3

,target,sredni_wiek,srednia_kwota
0,0,36.224286,2985.457143
1,1,33.963333,3938.126667


In [15]:
query4 = """
SELECT cel_kredytu, COUNT(*) as liczba
FROM klienci
WHERE target == 1
GROUP BY cel_kredytu
ORDER BY liczba DESC
"""

result4 = pd.read_sql_query(query4, conn)
result4

,cel_kredytu,liczba
0,A40,89
1,A43,62
2,A42,58
3,A49,34
4,A46,22
5,A41,17
6,A45,8
7,A410,5
8,A44,4
9,A48,1


In [16]:
query5 = """
SELECT wiek, kwota_kredytu, czas_trwania_msc, target
FROM klienci
WHERE target = 1 AND kwota_kredytu > 5000
ORDER BY kwota_kredytu DESC
LIMIT 10
"""

result5 = pd.read_sql_query(query5, conn)
result5

,wiek,kwota_kredytu,czas_trwania_msc,target
0,32,18424,48,1
1,58,15945,54,1
2,23,15672,48,1
3,68,14896,6,1
4,60,14782,60,1
5,23,14555,6,1
6,25,14421,48,1
7,57,14318,36,1
8,27,14027,60,1
9,38,12976,18,1


## Podsumowanie analizy SQL

Dane z datasetu German Credit zostały wczytane do bazy SQLite (`credit_risk.db`, tabela `klienci`),
a następnie przeanalizowane przy użyciu zapytań SQL jako uzupełnienie analizy wykonanej w pandas.

### Wykonane zapytania
1. Podstawowe statystyki opisowe (liczba klientów, średni wiek, średnia kwota kredytu)
2. Rozkład zmiennej target (liczba klientów którzy spłacili / nie spłacili)
3. Średni wiek i kwota kredytu w podziale na spłacalność (GROUP BY)
4. Najczęstszy cel kredytu wśród klientów, którzy nie spłacili (WHERE + GROUP BY + ORDER BY)
5. Top 10 klientów z najwyższą kwotą niespłaconego kredytu (WHERE z wieloma warunkami + LIMIT)

### Wnioski
- Wyniki zapytań SQL są spójne z obserwacjami z EDA wykonanego w pandas (m.in. rozkład targetu, 
  różnica średniego wieku między grupami), co potwierdza poprawność wcześniejszej analizy
- Zidentyfikowano cel kredytu najczęściej wiążący się z niespłaceniem - potencjalnie istotna 
  cecha przy dalszej analizie ryzyka
- **Ważne zastrzeżenie:** powyższe zapytania pokazują korelacje i wzorce w danych historycznych, 
  nie dowodzą związków przyczynowo-skutkowych. Wnioskowanie przyczynowe wymagałoby bardziej 
  zaawansowanych metod statystycznych 

### Wykorzystane elementy SQL
SELECT, WHERE, GROUP BY, ORDER BY, LIMIT, funkcje agregujące (COUNT, AVG), aliasy (AS), 
operator logiczny AND